# Cod3x Persona Trainer
Train LoRA adapter. Built by Codex Developer.

In [ ]:
# 1. Clone
import os
if not os.path.exists('Cod3x'):
    !git clone https://github.com/codexhaven/Cod3x.git
%cd Cod3x
!mkdir -p data persona_output


In [ ]:
# 2. Install
!pip install -q transformers datasets peft accelerate torch huggingface_hub


In [ ]:
# 3. HF Token
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded')
except:
    print('No HF_TOKEN secret')


In [ ]:
# 4. Load data
import json
with open('data/training_data.json') as f:
    data = json.load(f)
print(f'Loaded {len(data)} examples')


In [ ]:
# 5. OpenAssistant
from datasets import load_dataset
import json
ds = load_dataset('OpenAssistant/oasst1', split='train', streaming=True)
new = []
for i in ds:
    t = str(i.get('text',''))
    if 'ASSISTANT' in t:
        p = t.replace('[' ,'').replace(']','').replace(chr(34),'').split('ASSISTANT')
        if len(p)>=2:
            q = p[0].replace('USER','').strip()[:300]
            a = p[1].strip()[:500]
            if len(q)>10 and len(a)>20:
                new.append({'instruction':q,'response':a})
                if len(new)>=5000: break
print(f'Added {len(new)}')
with open('data/training_data.json') as f: old = json.load(f)
with open('data/training_data.json','w') as f: json.dump(old+new, f)
print(f'Total: {len(old)+len(new)}')


In [ ]:
# 6. Stack Exchange
from datasets import load_dataset
import json
ds = load_dataset('HuggingFaceH4/stack-exchange-preferences', split='train', streaming=True)
new = []
for i in ds:
    q = str(i.get('question','')).strip()[:300]
    a = str(i.get('answer','')).strip()[:800]
    if len(q)>20 and len(a)>50:
        new.append({'instruction':q,'response':a})
        if len(new)>=3000: break
print(f'Added {len(new)}')
with open('data/training_data.json') as f: old = json.load(f)
with open('data/training_data.json','w') as f: json.dump(old+new, f)
print(f'Total: {len(old)+len(new)}')


In [ ]:
# 7. Wikipedia
from datasets import load_dataset
import json, random
ds = load_dataset('wikipedia', '20220301.en', split='train', streaming=True)
qs = ['Tell me about {}','What is {}?','Explain {}','Describe {}']
new = []
for i in ds:
    t = str(i.get('title',''))
    tx = str(i.get('text',''))[:500]
    if len(tx)>100:
        q = random.choice(qs).format(t)
        new.append({'instruction':q,'response':f'I am Cod3x, built by Codex Developer. {t}: {tx}'})
        if len(new)>=3000: break
print(f'Added {len(new)}')
with open('data/training_data.json') as f: old = json.load(f)
with open('data/training_data.json','w') as f: json.dump(old+new, f)
print(f'Total: {len(old)+len(new)}')


In [ ]:
# 8. GSM8K Math
from datasets import load_dataset
import json
ds = load_dataset('gsm8k', 'main', split='train', streaming=True)
new = []
for i in ds:
    q = str(i.get('question','')).strip()
    a = str(i.get('answer','')).strip()
    if q and a:
        new.append({'instruction':q,'response':f'I am Cod3x, built by Codex Developer. {a}'})
        if len(new)>=2000: break
print(f'Added {len(new)}')
with open('data/training_data.json') as f: old = json.load(f)
with open('data/training_data.json','w') as f: json.dump(old+new, f)
print(f'Total: {len(old)+len(new)}')


In [ ]:
# 9. Train on CPU
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from transformers import Trainer, DataCollatorForLanguageModeling

BASE='Qwen/Qwen2.5-0.5B-Instruct'
OUT='./persona_output'
PNAME='cod3x-default'
print(f'Loading {BASE} on CPU...')
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32, device_map='cpu')
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token
with open('data/training_data.json') as f: data = json.load(f)
def fmt(ex):
    m=[{'role':'system','content':f'You are {PNAME}, trained by Cod3x.'},{'role':'user','content':ex['instruction']},{'role':'assistant','content':ex['response']}]
    return {'text':tok.apply_chat_template(m, tokenize=False)}
ds = Dataset.from_list(data).map(fmt)
ds = ds.map(lambda ex: tok(ex['text'], truncation=True, max_length=512), batched=True, remove_columns=['text'])
lc = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj','v_proj'], lora_dropout=0.1, bias='none', task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lc)
model.print_trainable_parameters()
ta = TrainingArguments(output_dir=OUT, per_device_train_batch_size=1, gradient_accumulation_steps=4, num_train_epochs=3, learning_rate=2e-4, logging_steps=10, save_strategy='epoch', report_to='none')
tr = Trainer(model=model, args=ta, train_dataset=ds, data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
print(f'Training on CPU with {len(data)} examples...')
tr.train()
model.save_pretrained(OUT)
tok.save_pretrained(OUT)
print(f'Persona saved to {OUT}')


In [ ]:
# 10. Upload
from huggingface_hub import login, upload_folder
login()
REPO='codexhaven/cod3x-persona'
upload_folder(folder_path='./persona_output', repo_id=REPO, repo_type='model')
print(f'Uploaded to https://huggingface.co/{REPO}')
